# Load the questions

In [1]:
import json
from dataclasses import dataclass
from typing import TypeAlias
import uuid

from model_0 import agent as agent_0
from model_1 import agent as agent_1

with open("benchmark.json") as f:
    qa = json.load(f)

qa = {"factual": qa.get("factual", [])}

qa = [
    (category, item)
    for category, items in qa.items()
    for item in items
]

E0000 00:00:1778460215.999521  929623 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1778460215.999565  929623 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1778460215.999566  929623 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1778460215.999569  929623 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1778460215.999570  929623 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the def

Found 0 new videos to add to collection youtube_videos_test_text.


Building KB: 0it [00:00, ?it/s]


Found 0 new videos to add to collection youtube_videos_test_hybrid.


Building KB: 0it [00:00, ?it/s]


# Load the architectures to evaluate
| Ablation              | Short-Term Memory |        KB        | Vector DB Indexing | Bias Analysis Sub-Agent |
|-----------------------|:-----------------:|:----------------:|:------------------:|:-----------------------:|
| 0. **Vanilla**        |         Y         |       N/A        |        N/A         |            N            |
| 1. **Text RAG**       |         Y         |       CSV        |        Text        |            N            |
| 2. **MM-RAG**         |         Y         | CSV + Thumbnails |       Hybrid       |            N            |
| 3. **Agentic MM-RAG** |         Y         | CSV + Thumbnails |       Hybrid       |            Y            |

In [ ]:
from tqdm.notebook import tqdm
from langchain.messages import HumanMessage


user_id = 25
agents = [
    (agent_0, uuid.uuid4().hex),
    (agent_1, uuid.uuid4().hex),
    # agent_2,
    # agent_3,
]

for category, item in tqdm(qa, desc="Question"):
    question = item["question"]
    golden_answer = item["answer"]

    print(f"\nQ:  {question}")
    print(f"G:  {golden_answer}")

    for i, (agent, thread_id) in enumerate(agents):
        result = agent.invoke(
            {"messages": [HumanMessage(content="Answer with only the final answer. No extra words beyond what explicitly asked: " + question)]},
            config={"configurable": {"thread_id": thread_id}},
            context={"user_id": user_id},
        )

        answer = result.get("messages", [])[-1].content
        item[f"model_{i}"] = answer

        print(f"A{i}: {answer}")

Question:   0%|          | 0/4 [00:00<?, ?it/s]


Q:  What is the exact watch date and time for the video titled 'Why Conservatives Are Under Siege'?
G:  2015-01-03 08:01:12
A0: I am sorry, but I cannot provide the exact watch date and time for the video titled 'Why Conservatives Are Under Siege' as that information is not available in the provided watch history data.


KeyboardInterrupt: 

In [4]:
from model_1 import agent as agent_1
from tqdm.notebook import tqdm
from langchain.messages import HumanMessage

user_id = 25
thread_id = uuid.uuid4().hex

result = agent_1.invoke(
    {"messages": [HumanMessage(content="Hello")]},
    config={"configurable": {"thread_id": thread_id}},
    context={"user_id": user_id},
)

In [ ]:
with open("benchmark_with_answers.json", "w") as f:
    json.dump(qa, f, indent=2, ensure_ascii=False)